# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lujain-Mahesar/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
print("Connected, ready to query.")

Connected, ready to query.


## Abstract
Some pages are ranking very high on Google search but receive virtually no
clicks, thus wasting their search visibility. I explored one month of the
FlyRank's actual search data (116,114 pages) in order to discover whether
I can somehow detect such pages. At first, I have attempted to create a
predictive model that failed to beat random guess baseline. However, then
I switched to a different method: instead of observing days
independently, I observed each page for an entire month, adding stability
of the page ranking feature. This helped to create an actual Random
Forest to beat the baseline (from 70.0% to 70.7%). Final result: 9,962
pages of high rank and traffic for a content team's attention.

## 1. Question

*The research question and the decision it supports.*

Which high ranking pages are secretly squandering their search engine
visibility because they get far fewer clicks than their ranking would
warrant, and could such an easy model accurately identify them?

This is important because the content team cannot possibly do a manual
review of all pages every month. A page that ranks well, but not clicking
through to get conversions is an untapped resource which more often than
not is simply a matter of metadata and can be fixed at a low cost.

What it would help make the decision on: Prioritizing limited review
hours of the content team to pages which fixing click through would
likely yield the most gain.

In [2]:
page_level = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        STDDEV(gsc_avg_position) AS position_std,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        COUNT(*) AS days_active
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) >= 50
""").df()

page_level["position_std"] = page_level["position_std"].fillna(0)
page_level["ctr"] = page_level["total_clicks"] / page_level["total_impressions"]

print(f"Total pages after aggregation: {len(page_level)}")
page_level.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages after aggregation: 116114


,content_hash_id,client_hash_id,avg_position,position_std,total_impressions,total_clicks,days_active,ctr
0,content_39d7361b4945d504,client_62f4a7e64f5e0096,4.074107,2.678448,77.0,0.0,24,0.000000
1,content_d49a012dcb924e31,client_62f4a7e64f5e0096,5.177774,2.109420,329.0,0.0,31,0.000000
2,content_614baf2af4330bd7,client_62f4a7e64f5e0096,4.685335,1.141410,772.0,1.0,31,0.001295
3,content_225dc9235023be5f,client_62f4a7e64f5e0096,17.148172,16.143845,488.0,1.0,31,0.002049
4,content_7dbc094b799e05a4,client_62f4a7e64f5e0096,5.956862,3.983587,705.0,1.0,31,0.001418


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Data source: FlyRank Internship Warehouse Release (v20260703), table
fact_content_daily_performance, filtered to month=2026-03 (a mid-panel
month, not the final sealed month, which is reserved as a held-out test
period per the dataset's own guidance).

I aggregated from daily rows up to one row per page for the whole month,
which gives more reliable numbers than any single day (a page's CTR on
one day can be noisy; a month's worth of clicks and impressions is much
more stable to judge).

After aggregation and filtering to pages with at least 50 total monthly
impressions (to avoid judging CTR off tiny, unreliable traffic), I'm left
with 116,114 unique pages.

Fields used: gsc_avg_position, gsc_impressions, gsc_clicks (all rolled up
to monthly averages/sums), plus a derived days_active count (how many days
in the month the page had any recorded impressions) and position_std (how
much a page's ranking fluctuated day-to-day).

Excluded: all GA4 fields (engaged_sessions, sessions, scroll_events),
these are only populated for about 3.7% of rows, so including them would
force me to drop 96%+ of the dataset, which isn't worth it for the modest
signal they might add. I also excluded client_hash_id from modeling
(kept only for grouping/reference), and no client names, domains, or URLs
appear anywhere in this notebook or its outputs.

In [3]:
ga4_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(ga4_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available  pct_available
0     9841378       413966.0            4.2


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
